In [14]:
import brainai, attentiveai, os, json, shutil
from glob import glob

In [6]:
arqs = sorted(glob('CSA_BETC Havas_Forebrain/**', recursive=True))
arqs = [a for a in arqs if '.' in a and a.split('.')[-1] in ['jpg', 'png', 'mp4']]

In [ ]:
# Rodar BRAIN A.I. e Attentive A.I.

brainai.initialize()
attentiveai.initialize()

for arq in arqs:
    destino = arq + '.json'
    results = json.load(open(destino, 'rt')) if os.path.exists(destino) else {}
    if 'engajamentoneural' in results and 'cognitivedemand' in results and 'focus' in results: continue

    print(arq)
    if not 'engajamentoneural' in results:
        if arq.endswith('.mp4'):
            tempos, curva = brainai.process_video(arq)
            results['tempos'] = tempos.tolist()
            results['engajamentoneural'] = curva.tolist()
        else:
            engajamentoneural = brainai.process_image(arq)
            results['engajamentoneural'] = engajamentoneural
    if not 'cognitivedemand' in results:
        if arq.endswith('.mp4'):
            results['cognitivedemand'] = attentiveai.calculate_cognitive_demand_index(arq)[1].tolist()
        else:
            results['cognitivedemand'] = attentiveai.calculate_cognitive_demand_index(arq)[0]
    if not 'focus' in results:
        if arq.endswith('.mp4'):
            results['focus'] = attentiveai.calculate_focus_index(arq)[1].tolist()
        else:
            results['focus'] = attentiveai.calculate_focus_index(arq)[0].tolist()

    json.dump(results, open(destino, 'wt'), indent=2)

brainai.release_resources()
attentiveai.release_resources()

In [15]:
# Isolar os arquivos .json (para ficar mais fácil de enviar para o Páris)

base_dir = 'CSA_BETC Havas_Forebrain'
json_dest_root = 'CSA_BETC Havas_Forebrain (jsons)'

json_files = sorted(glob(os.path.join(base_dir, '**', '*.json'), recursive=True))
for src_path in json_files:
    rel_path = os.path.relpath(src_path, base_dir)
    dest_path = os.path.join(json_dest_root, rel_path)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    shutil.copy2(src_path, dest_path)
